In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, col, trim

catalog = "mobility_dev"

source_path = (
    "abfss://landing@sturbanmobilitydev001.dfs.core.windows.net/"
    "reference/urban_mobility_sources/"
)

bronze_table = f"{catalog}.bronze.urban_mobility_sources_raw"
silver_table = f"{catalog}.silver.urban_mobility_sources"

In [0]:
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_path)
)

display(df_raw)
df_raw.printSchema()

In [0]:
df_bronze = (
    df_raw
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

display(spark.table(bronze_table))

In [0]:
df_silver = (
    spark.table(bronze_table)
    .select(
        trim(col("source_system")).alias("source_system"),
        trim(col("network_id")).alias("network_id"),
        trim(col("network_name")).alias("network_name"),
        trim(col("city")).alias("city"),
        trim(col("country")).alias("country"),
        col("latitude").cast("double").alias("latitude"),
        col("longitude").cast("double").alias("longitude"),
        col("station_count").cast("int").alias("station_count"),
        col("is_active").cast("boolean").alias("is_active"),
        col("load_date"),
        col("ingestion_timestamp"),
        col("source_file")
    )
    .where(col("network_id").isNotNull())
)

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

display(spark.table(silver_table))

In [0]:
spark.sql(f"""
SELECT 
    country,
    COUNT(*) AS source_count,
    SUM(station_count) AS total_stations
FROM {silver_table}
WHERE is_active = true
GROUP BY country
""").show()